# 01. Grid BFS

**Tier:** Deep. **Why you'd use this:** A grid of open and blocked cells needs the fewest steps from the top-left to the bottom-right. Every step into a neighbor costs one.

**After this lesson you can:** run BFS on a grid for unweighted shortest paths; see when Dijkstra replaces the queue.
About 25 minutes. **Uses:** [Loops](../../course/03-control-flow/02-loops-lesson.ipynb), [Lists](../../course/04-collections/01-lists-lesson.ipynb)


## Quick review

1. Why does BFS give shortest paths when every edge weighs one?
2. When do you reach for a heap instead of a queue?

<details><summary>Answers</summary>

Layer order explores distance `d` fully before `d + 1`, so the first time you reach the target is optimal. Use a heap (Dijkstra) when edge weights vary and can be greater than one.
</details>


## Recognize it

An unweighted graph or grid asks for fewest edges between two cells. Moves are up, down, left, right into open cells.

Signals: "shortest path", "minimum steps", grid with `0` open and `1` blocked. A look-alike that is not this pattern: minimum effort or weighted travel time — that needs Dijkstra (or binary search on effort), not plain BFS.

If the question is only "can you reach?" without a distance, DFS, BFS, or union-find all work — pick BFS when the length itself matters.


## Why it works this way

Breadth-first search stores a queue of cells with their distance from the start. A visited set (or a distance matrix) ensures each cell enters the queue at most once. Because every edge has weight one, the queue is sorted by distance automatically: you finish every cell at distance `d` before any cell at distance `d + 1`. The first time you dequeue the target, that distance is optimal.

Time is proportional to rows times columns: each open cell is processed once and looks at four neighbors. Memory is the queue plus the visited structure, worst case the whole grid. Depth-first search can dive down a long corridor before noticing a short route, so DFS does not give shortest paths. Dijkstra with a binary heap is the generalization when neighbor steps have different positive weights; the heap replaces the FIFO queue so the next pulled cell is the smallest tentative distance.

Mark visited when you enqueue, not when you dequeue. Marking late lets the same cell sit in the queue many times with longer paths and wastes work. Blocking walls and bounds checks keep neighbors inside the open grid. See [Lists](../../internals/03-lists/01-dynamic-array-lesson.ipynb) for the grid as nested lists, and prefer `collections.deque` for pops from the left.

The design reason for BFS here is the unit weight. When every step costs one, level order is already a priority order, so a heap would only add log factors.

Returning early on the first visit to the target is correct only because of that level order. On weighted graphs the first visit can be a long path that arrived early in wall-clock exploration order; you would need to relax distances instead. Keep that caveat glued to the unit-weight assumption so you do not export early-exit BFS into Dijkstra territory by habit.


## When to use it

Use grid BFS for fewest steps on an unweighted grid or graph. Prefer Dijkstra when weights vary. Prefer 0-1 BFS with a deque when weights are only 0 or 1. Prefer DFS or union-find when the question is reachability or components without distances. Prefer A* when you have a good heuristic and a huge state space.

If moves include diagonals or knight jumps, the neighbor list changes but the BFS skeleton stays. If every open cell must be visited for a multi-target answer, continue until the queue empties and record distances for all cells rather than exiting at the first target.


## Brute force, and why it's slow

Enumerate all simple paths with DFS and keep the shortest. Count path expansions.


In [1]:
def shortest_dfs_count(grid):
    rows, cols = len(grid), len(grid[0])
    best = None
    expansions = 0

    def walk(r, c, dist, seen):
        nonlocal best, expansions
        expansions += 1
        if (r, c) == (rows - 1, cols - 1):
            best = dist if best is None else min(best, dist)
            return
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 0 and (nr, nc) not in seen:
                seen.add((nr, nc))
                walk(nr, nc, dist + 1, seen)
                seen.remove((nr, nc))

    if grid[0][0] == 0:
        walk(0, 0, 1, {(0, 0)})
    return best, expansions


grid = [
    [0, 0, 0],
    [0, 1, 0],
    [0, 0, 0],
]
print(shortest_dfs_count(grid))


(5, 9)


## Key insight

Enqueue the start at distance one (or zero, if you count edges). The first time BFS reaches the target on a unit-weight grid is the shortest length. Mark cells when enqueued so each cell joins the queue once.


## Worked example

`grid[r][c] == 0` is open, `1` is blocked. Return the length of the shortest path from `(0, 0)` to `(rows - 1, cols - 1)` counting cells, or `-1` if none exists.


In [2]:
from collections import deque


def shortest_grid(grid: list[list[int]]) -> int:
    if not grid or grid[0][0] == 1:
        return -1
    rows, cols = len(grid), len(grid[0])
    # 1. Queue holds row, col, distance in cells.
    q = deque([(0, 0, 1)])
    seen = {(0, 0)}
    while q:
        r, c, dist = q.popleft()
        # 2. First arrival is shortest on unit weights.
        if (r, c) == (rows - 1, cols - 1):
            return dist
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 0 and (nr, nc) not in seen:
                # 3. Mark when enqueueing to avoid repeats.
                seen.add((nr, nc))
                q.append((nr, nc, dist + 1))
    return -1


print(shortest_grid([[0, 0, 0], [0, 1, 0], [0, 0, 0]]))
print(shortest_grid([[0, 1], [1, 0]]))


5
-1


## Trace

Grid:

```
0 0 0
0 1 0
0 0 0
```

| dequeue | dist | enqueue next |
|---|---|---|
| (0,0) | 1 | (1,0), (0,1) |
| (1,0) | 2 | (2,0) |
| (0,1) | 2 | (0,2) |
| (2,0) | 3 | (2,1) |
| (0,2) | 3 | (1,2) |
| (2,1) | 4 | (2,2) |
| (1,2) | 4 | — |
| (2,2) | **5** | return |

Answer **5**.


## Cost

Each open cell enters the queue once: time and memory grow with the number of cells. DFS over all paths can expand far more nodes on open grids. A binary-heap Dijkstra adds a log factor per edge when weights stop being uniform.


## Variants

Not solved here: [Rotting Oranges](../../challenges/problems/32-rotting-oranges.ipynb), [Network Delay Time](../../challenges/problems/49-network-delay-time.ipynb), [Clone Graph](../../challenges/problems/48-clone-graph.ipynb).


## Common mistake

This bug is silent. The search starts the distance at `0` while the problem counts cells on the path. The first time the end is reached the print is `4`, and the answer is `5`.


In [3]:

def shortest_grid_wrong(grid):
    if not grid or grid[0][0] == 1:
        return -1
    rows, cols = len(grid), len(grid[0])
    q = deque([(0, 0, 0)])  # starts at 0; should count the start cell as 1
    seen = {(0, 0)}
    while q:
        r, c, dist = q.popleft()
        if (r, c) == (rows - 1, cols - 1):
            return dist
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 0 and (nr, nc) not in seen:
                seen.add((nr, nc))
                q.append((nr, nc, dist + 1))
    return -1


print(shortest_grid_wrong([[0, 0, 0], [0, 1, 0], [0, 0, 0]]))


4


The fix starts the distance at `1` so the start cell counts. The worked example does that, so the same grid prints `5`.


In [4]:
print(shortest_grid([[0, 0, 0], [0, 1, 0], [0, 0, 0]]))


5


## Python idioms

`collections.deque` for BFS. For weighted grids, `heapq` with `(distance, r, c)` is Dijkstra. Store `dist[r][c]` and skip heap entries that are stale.


In [5]:
import heapq


def dijkstra_small(weights: list[list[int]]) -> int:
    rows, cols = len(weights), len(weights[0])
    dist = [[10**9] * cols for _ in range(rows)]
    dist[0][0] = weights[0][0]
    heap = [(dist[0][0], 0, 0)]
    while heap:
        cost, r, c = heapq.heappop(heap)
        if (r, c) == (rows - 1, cols - 1):
            return cost
        if cost > dist[r][c]:
            continue
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols:
                nxt = cost + weights[nr][nc]
                if nxt < dist[nr][nc]:
                    dist[nr][nc] = nxt
                    heapq.heappush(heap, (nxt, nr, nc))
    return -1


print(dijkstra_small([[1, 3, 1], [1, 5, 1], [4, 2, 1]]))


7


## Recap

- Unit weights: BFS first hit is shortest.
- Mark seen on enqueue.
- Weighted edges need a heap, not a plain queue.

Next: [practice](01-grid-bfs-practice.ipynb).
